# 🛰️ InSAR-SinkWatch: Scientific Exploration & Radar Analysis
### NASA International Space Apps Challenge 2026 | Team `visionspace_us`
**Challenge:** Dancing with the SARs (Tracking Earth's Surface Dynamics)
**Target Mission:** NASA-ISRO Synthetic Aperture Radar (NISAR L-band, $\lambda = 24.24\text{ cm}$)

---
This Jupyter Notebook documents the end-to-end scientific methodology behind **InSAR-SinkWatch**:
1. **InSAR Phase Formation:** Simulating repeat-pass interferometry over Ho Chi Minh City.
2. **Adaptive Filtering:** Goldstein InSAR filter to preserve structural fringes while suppressing phase noise.
3. **Phase-to-Displacement Physics:** Converting interferometric radians into millimeter-scale vertical velocity: $\Delta d = \frac{\lambda}{4\pi} \Delta \phi$.
4. **AI Anomaly Detection:** Isolating critical subsidence bowls ($< -20\text{ mm/year}$) and geotechnical infrastructure risk screening.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt

from src.pipeline.insar_processing import (
    NISAR_L_BAND_WAVELENGTH_MM,
    goldstein_phase_filter,
    phase_to_displacement_mm
)
from src.models.subsidence_detector import (
    detect_subsidence_hotspots,
    project_cumulative_subsidence,
    assess_infrastructure_vulnerability
)
from src.data_simulator import get_hcmc_insar_scene

print(f"✓ InSAR Pipeline loaded. NISAR L-band wavelength: {NISAR_L_BAND_WAVELENGTH_MM} mm ({NISAR_L_BAND_WAVELENGTH_MM/10} cm)")

## 1. Load Simulated InSAR Scene for Ho Chi Minh City
Calibrated according to empirical geotechnical data from the Ministry of Natural Resources and Environment (MONRE) and JICA.

In [ ]:
scene = get_hcmc_insar_scene(shape=(200, 200), seed=42)
raw_phase = scene["wrapped_phase"]
true_velocity = scene["velocity_map_mm_year"]
bounds = scene["bounds"]

print(f"Target Region: {scene['title']}")
print(f"Geographic Extent: Lon {bounds[0]} to {bounds[2]}, Lat {bounds[1]} to {bounds[3]}")
print(f"Peak Sinking Velocity: {np.min(true_velocity):.1f} mm/year")

## 2. InSAR Phase Filtering: Raw vs. Goldstein Filtered
The Goldstein filter applies adaptive frequency-domain smoothing to suppress decorrelation noise while maintaining sharp structural edges.

In [ ]:
filtered_phase = goldstein_phase_filter(raw_phase, alpha=0.5, window_size=5)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
im0 = axes[0].imshow(raw_phase, cmap='jet', vmin=-np.pi, vmax=np.pi)
axes[0].set_title('Raw InSAR Wrapped Phase [-π, +π]', fontsize=11)
axes[0].axis('off')
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(filtered_phase, cmap='jet', vmin=-np.pi, vmax=np.pi)
axes[1].set_title('Goldstein Filtered Phase [Clean Fringes]', fontsize=11)
axes[1].axis('off')
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## 3. Surface Velocity Field & AI Hotspot Detection
Classification into Stable ($> -10\text{ mm/yr}$), Moderate ($-10\text{ to } -20\text{ mm/yr}$), and Critical Subsidence Bowls ($< -20\text{ mm/yr}$).

In [ ]:
hotspots = detect_subsidence_hotspots(true_velocity, critical_threshold_mm=-20.0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
im_v = axes[0].imshow(true_velocity, cmap='turbo', vmin=-45, vmax=5)
axes[0].set_title('Vertical Deformation Velocity (mm/year)', fontsize=11)
axes[0].axis('off')
cbar = fig.colorbar(im_v, ax=axes[0], fraction=0.046, pad=0.04)
cbar.set_label('mm / year')

im_mask = axes[1].imshow(hotspots["classification_map"], cmap='viridis')
axes[1].set_title('AI Subsidence Risk Classification (0=Stable, 1=Mod, 2=Crit)', fontsize=11)
axes[1].axis('off')
fig.colorbar(im_mask, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

print(f"Critical Subsidence Pixels: {hotspots['critical_pixels']} pixels")
print(f"Maximum Subsidence Recorded: {hotspots['max_subsidence_rate_mm_year']} mm/year")

## 4. Multi-Year Geotechnical Projection (2026 - 2036)
Using Terzaghi soil consolidation compaction decay modeling to predict future settlement.

In [ ]:
years = list(range(1, 11))
cum_settlement = [np.min(project_cumulative_subsidence(true_velocity, forecast_years=y)) for y in years]

plt.figure(figsize=(8, 4.5))
plt.plot([2025 + y for y in years], np.abs(cum_settlement), marker='o', color='#f43f5e', linewidth=2.5)
plt.title('Projected Cumulative Subsidence at Core Depression Bowl (2026-2035)', fontsize=12)
plt.xlabel('Year')
plt.ylabel('Cumulative Settlement (mm)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()